# Part 1: Market capitalization vs. Book Value of Equity

In [2]:
%pip install --upgrade pip setuptools wheel
%pip install "pandas>=3.0" numpy matplotlib
%pip install sqlalchemy psycopg2-binary
%pip install wrds --no-deps

^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("walmart.csv")
data.head()

In [ ]:
data["sic"] = data["sic"].astype(str)

industry = data[data["sic"].str[:2] == "53"].copy()
industry["market_cap"] = industry["prcc_f"] * industry["csho"] #Market Cap = Share Price * Sharesutstanding

latest_year = industry[industry["tic"] == "WMT"]["fyear"].max()
print(latest_year)
latest = industry[industry["fyear"] == latest_year].copy()
latest = latest.dropna(subset=["market_cap"])
latest["market_cap"].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
)

In [ ]:
walmart = latest[latest["tic"] == "WMT"]
wmt_market_cap = walmart["market_cap"].iloc[0] #iloc takes the first value of the market cap column
print("Walmart's Market Capitalisation:", wmt_market_cap)

In [ ]:
#find the difference in market caps
#for every company in 'latest', company market cap - walmart market cap
# abs makes the result positive
latest["difference"] = abs(
    latest["market_cap"] - wmt_market_cap
)
not_walmart=latest[latest["tic"] != "WMT"] #removes walmart
sorted_companies=not_walmart.sort_values("difference") #sorts company market cap difference in ascending order
peers = sorted_companies.head(5) #keeps first 5 companies (closest peers)
print(peers[["conm", "tic", "market_cap", "difference"]])


In [ ]:
#calculate book-to-market=book value of equity(common equity)/market value of equity(market cap)
industry["book_to_market"] = (
    industry["ceq"] / industry["market_cap"]
)
companies = ["WMT"] + peers["tic"].tolist()
print(companies)
comparison = industry[industry["tic"].isin(companies)].copy()
years = list(range(int(latest_year) - 4, int(latest_year) + 1))

print(years)
comparison = comparison[comparison["fyear"].isin(years)].copy()

btm_table = comparison.pivot_table(
    index="fyear",
    columns="tic",
    values="book_to_market"
)

print(btm_table.round(3))

plt.figure(figsize=(10, 6))

for company in companies:
    company_data = comparison[comparison["tic"] == company]
    company_data = company_data.sort_values("fyear")

    plt.plot(
        company_data["fyear"],
        company_data["book_to_market"],
        marker="o",
        label=company
    )

plt.xlabel("Fiscal Year")
plt.ylabel("Book-to-Market Ratio")
plt.title("Book-to-Market Ratio: Walmart vs Peers")
plt.legend()
plt.grid()

plt.show()


In [ ]:
btm_change = btm_table.iloc[-1] - btm_table.iloc[0]

print(btm_change.round(3)) #2025 B/M - 2021 B/M

comparison = industry[industry["tic"].isin(companies)].copy()

years = list(range(int(latest_year) - 4, int(latest_year) + 1))

comparison = comparison[
    comparison["fyear"].isin(years)
].copy()

btm_table = comparison.pivot_table(
    index="fyear",
    columns="tic",
    values="book_to_market"
)

print(btm_table.round(3))

btm_change = btm_table.iloc[-1] - btm_table.iloc[0]

print(btm_change.round(3))

### Book-to-Market Comparison

Walmart's book-to-market ratio fell from **0.216 in 2021** to **0.105 in 2025**, representing a decrease of **0.111** over the five-year period.

This contrasts with most of its selected peers, whose book-to-market ratios increased over the same period. **Costco** was the only other peer to experience a decline, although its decrease was much smaller at **0.018**.

By 2025, Walmart's book-to-market ratio of **0.105** was lower than those of **Dollar General, Target, and Wal-Mart de México**, but higher than **Costco and Dollarama**.

# Part 2 Profitability

In [ ]:
import pandas as pd

# Read data
df = pd.read_csv("financial_data.csv")

# Calculate ratios
df["ROA"] = (df["ni"] / df["at"]) * 100
df["ROE"] = (df["ni"] / df["seq"]) * 100

# Show results
print(df[["tic", "fyear", "ROA", "ROE"]])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("financial_data.csv")
df["ROA"] = (df["ni"] / df["at"]) * 100
df["ROE"] = (df["ni"] / df["seq"]) * 100

ASSIGNED = "COST"  # your assigned company

# Keep the last five years
df5 = df[df["fyear"] > df["fyear"].max() - 5]
print(df5[["tic", "fyear", "ROA", "ROE"]].sort_values(["tic", "fyear"]))

# Plot ROA and ROE: assigned company vs peers
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ["ROA", "ROE"]):
    for tic, g in df5.groupby("tic"):
        g = g.sort_values("fyear")
        if tic == ASSIGNED:
            ax.plot(g["fyear"], g[metric], marker="o", linewidth=3, label=tic)
        else:
            ax.plot(g["fyear"], g[metric], marker="o", linestyle="--", alpha=0.7, label=tic)
    ax.set_title(f"{metric} (%), last five years")
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel(f"{metric} (%)")
    ax.set_xticks(sorted(df5["fyear"].unique()))
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("roa_roe.png", dpi=200)
plt.show()

In [ ]:
import pandas as pd

# Read data
df = pd.read_csv("financial_data2.csv")

# Calculate ratios
df["ROA"] = (df["ni"] / df["at"]) * 100
df["ROE"] = (df["ni"] / df["seq"]) * 100

# Show results
print(df[["tic", "fyear", "ROA", "ROE"]])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("financial_data2.csv")
df["ROA"] = (df["ni"] / df["at"]) * 100
df["ROE"] = (df["ni"] / df["seq"]) * 100

ASSIGNED = "COST"  # your assigned company

# Keep the last five years
df5 = df[df["fyear"] > df["fyear"].max() - 5]
print(df5[["tic", "fyear", "ROA", "ROE"]].sort_values(["tic", "fyear"]))

# Plot ROA and ROE: assigned company vs peers
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ["ROA", "ROE"]):
    for tic, g in df5.groupby("tic"):
        g = g.sort_values("fyear")
        if tic == ASSIGNED:
            ax.plot(g["fyear"], g[metric], marker="o", linewidth=3, label=tic)
        else:
            ax.plot(g["fyear"], g[metric], marker="o", linestyle="--", alpha=0.7, label=tic)
    ax.set_title(f"{metric} (%), last five years")
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel(f"{metric} (%)")
    ax.set_xticks(sorted(df5["fyear"].unique()))
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("roa_roe.png", dpi=200)
plt.show()

Walmart’s profitability was below its peers for most of the period but has largely caught up. In 2021 its ROA (5.6%) and ROE (16.4%) were well below the peer averages (10.4% and 36.2%). Since 2022, Walmart’s ROA and ROE have risen steadily, while the peer averages have fallen, so by 2025 the gaps are almost closed (ROA 7.7% vs 7.9%, ROE 22.0% vs 22.4%).

The main difference is direction. Walmart improved, while Dollar General and Target saw sharp declines after 2021-2022 and WMMVY slipped from 2023. Costco was the only peer whose returns stayed strong, and it remains ahead of Walmart on ROA (10.5% vs 7.7%). Walmart’s low ROA reflects its very large asset base, but its net income has grown faster than its assets since 2022.

# Part 3: Stock-Market Performance

In [ ]:
%pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [ ]:
crsp = pd.read_csv("crsp.csv")
ff = pd.read_csv("ff.csv")

crsp.head()

In [ ]:
ff.head()

In [ ]:
# turn the date columns into proper dates
crsp["date"] = pd.to_datetime(crsp["DlyCalDt"])
ff["date"] = pd.to_datetime(ff["date"])

# quick check of which companies we have
crsp["Ticker"].unique()

In [ ]:
# one column of returns per company
rets = crsp.pivot(index="date", columns="Ticker", values="DlyRet")

# put the returns and factors side by side, matched on date
data = pd.merge(rets, ff, on="date")
data = data.set_index("date").dropna()

data.head()

In [ ]:
# check the sample period
print(data.index.min(), data.index.max(), len(data))

In [ ]:
tickers = ["WMT", "COST", "TGT", "KR"]

# Market return = market excess return + risk-free rate
data["MKT"] = data["mktrf"] + data["rf"]
all_rets = data[tickers + ["MKT"]]

summary = pd.DataFrame({
    "Mean daily return": all_rets.mean(),
    "Annualised return": all_rets.mean() * 252,
    "Annualised volatility": all_rets.std() * np.sqrt(252),
    "Total 5-year return": (1 + all_rets).prod() - 1,
})
print(summary.round(4))

In [ ]:
data[tickers].plot(subplots=True, figsize=(10, 9), sharey=True, title="Daily returns")
plt.savefig("fig1_daily_returns.png")
plt.show()

In [ ]:
annual = (1 + all_rets).groupby(all_rets.index.year).prod() - 1
print(annual.round(4))

annual.plot(kind="bar", figsize=(10, 5), title="Calendar-year returns")
plt.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig("fig2_annual_returns.png", dpi=200)
plt.show()

In [ ]:
cum = (1 + all_rets).cumprod() - 1

cum.plot(figsize=(10, 5), title="Cumulative returns, 2021–2025")
plt.axhline(0, color="black", linewidth=0.5)
plt.savefig("fig3_cumulative_returns.png")
plt.show()

In [ ]:
capm_results = {}

for t in tickers:
    y = data[t] - data["rf"]               # excess stock return
    X = sm.add_constant(data[["mktrf"]])   # market excess return + intercept
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

    capm_results[t] = {
        "Alpha (daily)": model.params["const"],
        "Alpha (annualised)": model.params["const"] * 252,
        "Alpha t-stat": model.tvalues["const"],
        "Alpha p-value": model.pvalues["const"],
        "Beta": model.params["mktrf"],
        "R-squared": model.rsquared,
    }

capm_table = pd.DataFrame(capm_results).T
print(capm_table.round(4))

In [ ]:
ff3_results = {}

for t in tickers:
    y = data[t] - data["rf"]
    X = sm.add_constant(data[["mktrf", "smb", "hml"]])
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

    ff3_results[t] = {
        "Alpha (annualised)": model.params["const"] * 252,
        "Alpha t-stat": model.tvalues["const"],
        "Alpha p-value": model.pvalues["const"],
        "Beta (MKT)": model.params["mktrf"],
        "SMB": model.params["smb"],
        "HML": model.params["hml"],
        "R-squared": model.rsquared,
    }

ff3_table = pd.DataFrame(ff3_results).T
print(ff3_table.round(4))

In [ ]:
comparison = pd.DataFrame({
    "Raw return (ann.)": summary.loc[tickers, "Annualised return"],
    "CAPM alpha (ann.)": capm_table["Alpha (annualised)"],
    "CAPM t-stat": capm_table["Alpha t-stat"],
    "FF3 alpha (ann.)": ff3_table["Alpha (annualised)"],
    "FF3 t-stat": ff3_table["Alpha t-stat"],
})
print(comparison.round(4))
comparison.round(4).to_csv("comparison_table.csv")

comparison[["Raw return (ann.)", "CAPM alpha (ann.)", "FF3 alpha (ann.)"]].plot(
    kind="bar", figsize=(10, 5), title="Raw returns vs risk-adjusted alphas (annualised)")
plt.axhline(0, color="black", linewidth=0.5)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.savefig("fig4_raw_vs_alpha.png", dpi=200)
plt.show()

# Part 4: Earnings Announcements

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

EVENT_DATE  = pd.Timestamp("2025-11-20")
EVENT_FILE  = "wmt_event.csv"
VOLUME_FILE = "wmt_volume.csv"



# WRDS Event Study Data
ev = pd.read_csv(EVENT_FILE) # reads the file
ev = ev[["evttime", "ret_m", "abret_m", "car_m"]] #keeps only these columns
ev = ev.rename(columns={"evttime": "rel", "ret_m": "ret", "abret_m": "abret", "car_m": "car"}) #rename columns

# WRDS Compustat Data
px = pd.read_csv(VOLUME_FILE) #read the volume file
px.columns = [c.lower() for c in px.columns] #lowers the column names

#Alternatively
#(new_names = []
#for c in px.columns:
#   new_names.append(c.lower())

px = px.rename(columns={"dlycaldt": "date", "dlyvol": "vol"}) #rename colums

px["date"] = pd.to_datetime(px["date"]) #turns text into date format
px = px.sort_values("date").reset_index(drop=True) #puts the rows in date order, oldest first. (might not be in order, renumbers the rows 0, 1, 2, 3 and so on. Sorting shuffles the row numbers along with the rows, so you’d see something like 5, 2, 9. To make this clean, reset the index to 0, 1, 2.), drop duplicates

#We do this to match volume data with the event study data. The event study data is in relative days, so we need to create a relative day column in the volume data as well.
pos0 = px.index[px["date"] == EVENT_DATE][0] #find the position of the event date in the index. This is a list, so we take the first element with [0], this a boolean True or False so only one date should remain which is our event date.

#print(px.head()) to verify postion vs relative position
px["rel"] = px.index - pos0 #creates a new column called rel, which is the index minus the position of the event date. This gives us a relative day count, with 0 being the event date, negative numbers before, and positive numbers after.

base = px[(px["rel"] >= -20) & (px["rel"] <= -1)]["vol"].mean()  #calculates the average volume for the 20 days before the event date, which we will use as a baseline for comparison in the volume chart.

df = ev.merge(px[["rel", "date", "vol"]], on="rel", how="left") #matches the two dataframes on the relative day column, keeping all rows from the event study data and adding the volume data where available.
print(df) #check if worked

#Creates the graphs and charts
fig, ax = plt.subplots(3, 1, figsize=(11, 11), sharex=True)

ax[0].bar(df["rel"], df["ret"] * 100, color="tab:blue")
ax[0].set_ylabel("Raw return (%)")
ax[0].set_title("Walmart (WMT): earnings announcement 20 Nov 2025, days -20 to +20")

ax[1].bar(df["rel"], df["abret"] * 100, color="tab:orange", label="Abnormal return")
ax[1].set_ylabel("Abnormal return (%)")
ax2 = ax[1].twinx()
ax2.plot(df["rel"], df["car"] * 100, color="black", marker="o", ms=3, label="CAR")
ax2.set_ylabel("CAR (%)")
h1, l1 = ax[1].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax[1].legend(h1 + h2, l1 + l2, loc="upper left")

ax[2].bar(df["rel"], df["vol"] / 1e6, color="tab:green")
ax[2].axhline(base / 1e6, color="red", ls="--", label="Average volume, days -20 to -1")
ax[2].set_ylabel("Volume (millions of shares)")
ax[2].set_xlabel("Trading days relative to announcement (0 = 20 Nov 2025)")
ax[2].legend()

for a in ax:
    a.axvline(0, color="grey", ls=":")

plt.tight_layout()
plt.savefig("walmart_event_study.png", dpi=150)
plt.show()

